In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.layers import Dense
import numpy as np
import re
import scipy.io

infile_path = 'Input5dB.csv'
df = pd.read_csv(infile_path)


# Read the text file into a DataFrame without headers
# - delim_whitespace=True handles any whitespace (spaces, tabs) as delimiters
# - header=None indicates that the file does not have a header row
# - names=['time', 'xI_real', 'xI_imag'] assigns column names manually
# df = pd.read_csv(
#     infile_path,
#     delim_whitespace=True,
#     header=None,
#     names=['time', 'xI_real', 'xI_imag']
# )

# Extract the 'xI_real' and 'xI_imag' columns
x = df[['xI_real', 'xI_imag']]

# Display the full DataFrame to verify
#x_trimmed = x.iloc[:7001].reset_index(drop=True)

# Verify the new size
#print(f"Number of samples in x_trimmed: {len(x_trimmed)}")

X = x;

# Load the training data
outfile_path = 'IQout_5dB.csv'
df = pd.read_csv(outfile_path)

y = df[['yout_real', 'yout_imag']]

print("Shape of y:", y.shape)
print("Shape of X:", X.shape)

print('NN input files recieved')

# print(f'Output saved to {output_mat_file_path}')

2025-03-31 09:46:12.505430: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-31 09:46:12.506678: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-31 09:46:12.514053: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-31 09:46:12.531692: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743432372.561337 2134980 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743432372.57

FileNotFoundError: [Errno 2] No such file or directory: 'Input5dB.csv'

In [3]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Build the model
model = Sequential([
    Input(shape=(2,)),  # Explicitly defining the input shape
    Dense(32, activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(2)  # Output layer with 2 neurons for xI_real and xI_imag
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=8, validation_split=0.2, verbose=1)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)


y_pred = model.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f'Test Loss: {test_loss}')
print(f'Test accuracy: {test_accuracy}')
print(f'Test RMSE: {rmse}')

print("Training complete")

# Function to parse complex numbers, including those with scientific notation from excel csv file
def parse_complex_number(s):
    s = s.replace("i", "j")  
    # Updated regex to handle scientific notation
    match = re.match(r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([-+]\d*\.?\d+(?:[eE][-+]?\d+)?)j", s.replace(" ", ""))
    if match:
        real_part = float(match.group(1))
        imag_part = float(match.group(2))
        return real_part, imag_part
    else:
        raise ValueError(f"Malformed complex number: {s}")


# # Loop through files numbered 1 to 31
for i in range(1, 42):
    # Load the CSV file for predictions
    predict_file_path = f'/home/luke_mccubbin1/python_scripts/0210Dataset/sinewaves/sine_test_{i}.csv'
    df_predict = pd.read_csv(predict_file_path)

    # Separate the real and imaginary parts of the 'yout' complex numbers
    df_predict[['xI_real', 'xI_imag']] = df_predict['xI'].apply(
        lambda x: pd.Series(parse_complex_number(x))
    )

    # Prepare input features for prediction
    X_new = df_predict[['xI_real', 'xI_imag']]

    # Predict using the model
    y_new_pred = model.predict(X_new)

    # Store predictions back into the DataFrame
    df_predict['yout_predict_real'] = y_new_pred[:, 0]
    df_predict['yout_predict_imag'] = y_new_pred[:, 1]

    # Combine real and imaginary parts into a complex number
    df_predict['yout_predict'] = (df_predict['yout_predict_real'] + 1j * df_predict['yout_predict_imag']).astype(np.complex128)

    # Prepare data for saving as MAT file
    data_to_save = {'yout_predict': df_predict['yout_predict'].values}

    # Save to a MAT file with a dynamic filename
    output_mat_file_path = f'/home/luke_mccubbin1/python_scripts/0210Dataset/sineresults5dB/sine_result_{i}.mat'
    scipy.io.savemat(output_mat_file_path, data_to_save)

    print(f'Predictions saved to {output_mat_file_path}')

Epoch 1/8
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9621 - loss: 0.5360 - val_accuracy: 0.9855 - val_loss: 0.0119
Epoch 2/8
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9871 - loss: 0.0115 - val_accuracy: 0.9863 - val_loss: 0.0106
Epoch 3/8
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9868 - loss: 0.0115 - val_accuracy: 0.9851 - val_loss: 0.0112
Epoch 4/8
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9870 - loss: 0.0114 - val_accuracy: 0.9851 - val_loss: 0.0117
Epoch 5/8
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9859 - loss: 0.0113 - val_accuracy: 0.9845 - val_loss: 0.0128
Epoch 6/8
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9866 - loss: 0.0113 - val_accuracy: 0.9857 - val_loss: 0.0102
Epoch 7/8
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9873 - loss: 0.0113 - val_accuracy: 0.9829 - val_loss: 0.0122
Epoch 8/8
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9866 - loss: 0.0111 - val_accu